# 04 · Amihud Illiquidity Ratio
**Pillar (ii) Investability — Liquidity**

$$ILLIQ_{i,t} = \frac{|R_{i,t}|}{V_{i,t}} \qquad \text{Index ILLIQ} = \sum_j w_j \cdot \overline{ILLIQ}_j$$

- $|R_{i,t}|$ = absolute daily return of stock $i$
- $V_{i,t}$ = daily traded value (JPY) = close × volume
- Winsorisation at 1 / 99 % across the flattened stock-day matrix before time-averaging

Same-window counterfactual (2025-26 robustness window): per-stock ILLIQ is identical across the two indices; the difference in index-level ILLIQ is driven entirely by membership and weighting. The kept-vs-removed test measures whether the reform preferentially dropped the least-liquid pre-reform names.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# --- Indices (same-window counterfactual) -----------------------------------
orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']  / pf['FF_MktCap'].sum()

# --- Robust-window CLOSE/VOLUME (main + gap) --------------------------------
def _load_robust(kind):
    main_fp = os.path.join(DATA_DIR, f'{kind}_robust.parquet')
    gap_fp  = os.path.join(DATA_DIR, f'{kind}_robust_gap.parquet')
    frames = []
    if os.path.exists(main_fp):
        frames.append(pd.read_parquet(main_fp))
    if os.path.exists(gap_fp):
        frames.append(pd.read_parquet(gap_fp))
    df = pd.concat(frames, axis=1) if frames else pd.DataFrame()
    return df.loc[:, ~df.columns.duplicated()]

df_close  = _load_robust('prices')
df_volume = _load_robust('volume')

print(f'Original-today : {len(orig):,} stocks    Pro-forma-today : {len(pf):,} stocks')
print(f'Robust CLOSE : {df_close.shape}   VOLUME : {df_volume.shape}')


In [ ]:
# ── Per-stock Amihud ILLIQ on the 2025-26 window (ONE computation) ───────────
def amihud(price_df, volume_df, lo=0.01, hi=0.01):
    price_df  = price_df.apply(pd.to_numeric, errors='coerce')
    volume_df = volume_df.apply(pd.to_numeric, errors='coerce')
    ret   = price_df.pct_change()
    tv    = price_df * volume_df
    illiq = (ret.abs() / tv).replace([np.inf, -np.inf], np.nan)
    flat  = pd.to_numeric(pd.Series(illiq.values.flatten()), errors='coerce').dropna().values
    if len(flat) == 0:
        return pd.Series(dtype=float, index=price_df.columns, name='ILLIQ')
    lo_val = np.percentile(flat, lo * 100)
    hi_val = np.percentile(flat, (1 - hi) * 100)
    illiq  = illiq.clip(lower=lo_val, upper=hi_val)
    return illiq.mean(axis=0)

print('Computing Amihud ILLIQ on the 2025-26 robust window ...')
illiq_stock = amihud(df_close, df_volume)
illiq_stock.name = 'ILLIQ'
print(f'  Stocks with ILLIQ: {illiq_stock.notna().sum()} / {len(illiq_stock)}')

# Merge the SAME per-stock ILLIQ onto both frames.
orig = orig.merge(pd.DataFrame({'RIC': illiq_stock.index, 'ILLIQ': illiq_stock.values}),
                  on='RIC', how='left')
pf   = pf.merge(pd.DataFrame({'RIC': illiq_stock.index, 'ILLIQ': illiq_stock.values}),
                on='RIC', how='left')

illiq_idx_orig = (orig['w'] * orig['ILLIQ'].fillna(0)).sum()
illiq_idx_pf   = (pf['w']   * pf['ILLIQ'].fillna(0)).sum()
pct_chg = (illiq_idx_pf - illiq_idx_orig) / illiq_idx_orig * 100

print('=' * 55)
print(f'  Index ILLIQ  Original-today  : {illiq_idx_orig:.4e}')
print(f'  Index ILLIQ  Pro-forma-today : {illiq_idx_pf:.4e}')
print(f'  Change (PF − Orig) / Orig    : {pct_chg:+.2f}%')
print('=' * 55)


In [ ]:
# ── Kept vs removed test ─────────────────────────────────────────────────────
# Kept:    RICs present in BOTH original-today and pro-forma (pre-reform
#          members that survived the reform).
# Removed: RICs present in original-today but NOT in pro-forma (pre-reform
#          members dropped by the reform).
pf_rics  = set(pf['RIC'])
kept     = orig[orig['RIC'].isin(pf_rics)]['ILLIQ'].dropna()
removed  = orig[~orig['RIC'].isin(pf_rics)]['ILLIQ'].dropna()

print(f'N kept    : {len(kept):,}')
print(f'N removed : {len(removed):,}')

t_stat, p_t  = stats.ttest_ind(removed, kept, equal_var=False)
u_stat, p_mw = stats.mannwhitneyu(removed, kept, alternative='greater')

print(f'\nH₀: ILLIQ(removed) ≤ ILLIQ(kept)')
print(f'  Welch t-test : t={t_stat:.4f}  p={p_t:.4e}')
print(f'  Mann-Whitney : U={u_stat:.0f}  p={p_mw:.4e}')
print(f'  Median ILLIQ removed : {removed.median():.4e}')
print(f'  Median ILLIQ kept    : {kept.median():.4e}')
if kept.median() > 0:
    print(f'  Ratio removed/kept   : {removed.median()/kept.median():.2f}x')


In [ ]:
# ── (Removed — the old cell_030 computed a separate robust-window ILLIQ.      ─
# Since per-stock ILLIQ is now computed once on the 2025-26 window in cell_010,
# both indices are already measured on the robustness window. No second pass
# is needed.)
print('(cell_030 intentionally blank — per-stock ILLIQ already on 2025-26 window.)')


In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A: index ILLIQ bar
ax = axes[0, 0]
labels = ['Original-today', 'Pro-forma-today']
values = [illiq_idx_orig, illiq_idx_pf]
colors = ['steelblue', 'darkorange']
bars = ax.bar(labels, values, color=colors, edgecolor='black', alpha=0.85)
ax.bar_label(bars, fmt='{:.2e}', padding=3, fontsize=8)
ax.set_ylabel('Index ILLIQ')
ax.set_title(f'A  Index-Weighted Amihud ILLIQ  (Δ={pct_chg:+.1f}%)')

# B: distribution (log10)
ax = axes[0, 1]
ax.hist(np.log10(orig['ILLIQ'].dropna().clip(1e-20)), bins=60,
        alpha=0.6, label='Original-today',  color='steelblue')
ax.hist(np.log10(pf['ILLIQ'].dropna().clip(1e-20)),   bins=60,
        alpha=0.6, label='Pro-forma-today', color='darkorange')
ax.set_xlabel('log₁₀(ILLIQ)')
ax.set_title('B  Per-stock ILLIQ Distribution')
ax.legend()

# C: ECDF
ax = axes[1, 0]
for label, s, c in [('Original-today',  orig['ILLIQ'].dropna(), 'steelblue'),
                    ('Pro-forma-today', pf['ILLIQ'].dropna(),   'darkorange')]:
    sv = np.sort(s)
    if len(sv):
        ax.plot(np.log10(sv.clip(1e-20)), np.arange(1, len(sv)+1)/len(sv), label=label, color=c)
ax.set_xlabel('log₁₀(ILLIQ)')
ax.set_ylabel('Cumulative prob.')
ax.set_title('C  ECDF')
ax.legend()

# D: kept vs removed boxplot
ax = axes[1, 1]
bp = ax.boxplot([np.log10(kept.clip(1e-20)), np.log10(removed.clip(1e-20))],
                patch_artist=True, notch=True,
                medianprops=dict(color='black', lw=2))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('tomato')
ax.set_xticklabels(['Kept', 'Removed'])
ax.set_ylabel('log₁₀(ILLIQ)')
ax.set_title('D  Kept vs Removed (pre-reform members)')
sig = '***' if p_mw<0.01 else '**' if p_mw<0.05 else '*' if p_mw<0.1 else 'n.s.'
ax.annotate(f'Mann-Whitney (one-sided, removed>kept)\np={p_mw:.2e} {sig}',
            xy=(0.5, 0.95), xycoords='axes fraction', ha='center',
            bbox=dict(boxstyle='round', fc='lightyellow', ec='gray'))

plt.suptitle('Amihud Illiquidity — TOPIX Reform (same-window counterfactual, 2025-26)', fontsize=12)
plt.tight_layout()
plt.savefig('amihud_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: amihud_analysis.png')


In [ ]:
summary = pd.DataFrame({
    'Metric': ['N stocks', 'Index ILLIQ (2025-26)',
               'Median stock ILLIQ', 'P90 stock ILLIQ',
               'Kept vs removed — Mann-Whitney p (removed > kept)'],
    'Original-today':  [f'{len(orig):,}', f'{illiq_idx_orig:.4e}',
                        f'{orig["ILLIQ"].median():.4e}', f'{orig["ILLIQ"].quantile(0.9):.4e}', ''],
    'Pro-forma-today': [f'{len(pf):,}',   f'{illiq_idx_pf:.4e}',
                        f'{pf["ILLIQ"].median():.4e}',   f'{pf["ILLIQ"].quantile(0.9):.4e}',
                        f'{p_mw:.4e}']
})
print(summary.to_string(index=False))
summary.to_csv('amihud_summary.csv', index=False)
print('\nSaved: amihud_summary.csv')
